# 1. Búsqueda informada: material para el taller

Para resolver el taller posterior, primero consolidamos las piezas de la búsqueda informada.

- $g(n)$: costo acumulado desde el inicio hasta $n$.
- $h(n)$: estimación del costo desde $n$ hasta el objetivo.
- En A*: $f(n)=g(n)+h(n)$.

**Costo Uniforme:** prioriza menor $g(n)$.  
**A\*:** prioriza menor $g(n)+h(n)$.  
**Beam Search:** conserva solo los mejores $k$ candidatos de cada nivel.

## 1.1 Entorno

In [ ]:
from heapq import heappush, heappop
from itertools import count
import math
import matplotlib.pyplot as plt

print("Entorno listo")

## 1.2 Heurística

Costo Uniforme solo utiliza información sobre el costo ya recorrido: $g(n)$. Para orientar la búsqueda hacia el objetivo incorporamos una estimación del costo mínimo restante desde el nodo $n$ hasta el objetivo:

$$h(n)$$

La heurística $h(n)$ intenta aproximar el costo real mínimo restante $h^*(n)$:

$$h(n)\approx h^*(n)$$

Una heurística es **admisible** cuando nunca sobreestima el costo real:

$$\boxed{h(n)\leq h^*(n)}$$

> **La heurística no tiene que acertar exactamente. Tiene que estimar.**  
> Una heurística demasiado débil puede aportar poca información. Una heurística que sobreestima puede orientar de manera agresiva la búsqueda, pero pierde garantías de optimalidad.

## 1.3 A*

A* combina costo acumulado y estimación:

$$\boxed{f(n)=g(n)+h(n)}$$

Se expande primero el nodo con menor $f(n)$.

## 1.4 Caso aplicado: cuadrícula

Trabajaremos con un mapa:

- `0`: celda libre
- `1`: obstáculo
- acciones: arriba, abajo, izquierda y derecha
- costo de cada movimiento: 1

Para A* usaremos distancia Manhattan:

$$h(n)=|x_n-x_g|+|y_n-y_g|$$

In [ ]:
mapa = [
    [0,0,0,0,0,0,0],
    [0,1,1,1,0,1,0],
    [0,0,0,1,0,1,0],
    [1,1,0,1,0,0,0],
    [0,0,0,0,1,1,0],
    [0,1,1,0,0,0,0],
    [0,0,0,0,1,0,0],
]
INICIO=(0,0); OBJETIVO=(6,6)
MOVIMIENTOS=[(1,0),(-1,0),(0,1),(0,-1)]

def sucesores_mapa(mapa,pos):
    filas=len(mapa); cols=len(mapa[0]); f,c=pos; out=[]
    for df,dc in MOVIMIENTOS:
        nf,nc=f+df,c+dc
        if 0<=nf<filas and 0<=nc<cols and mapa[nf][nc]==0:
            out.append((nf,nc))
    return out

def manhattan(a,b):
    return abs(a[0]-b[0])+abs(a[1]-b[1])

### 1.4.1 Costo Uniforme

In [ ]:
def ucs_mapa(mapa,inicio,objetivo):
    tie=count(); frontera=[(0,next(tie),inicio,[inicio])]
    mejor_g={inicio:0}; expandidos=0
    while frontera:
        g,_,nodo,camino=heappop(frontera)
        if g != mejor_g.get(nodo): continue
        expandidos += 1
        if nodo==objetivo:
            return {"camino":camino,"costo":g,"expandidos":expandidos}
        for vecino in sucesores_mapa(mapa,nodo):
            ng=g+1
            if ng < mejor_g.get(vecino, math.inf):
                mejor_g[vecino]=ng
                heappush(frontera,(ng,next(tie),vecino,camino+[vecino]))
    return None

resultado_ucs_mapa=ucs_mapa(mapa,INICIO,OBJETIVO)
resultado_ucs_mapa

### 1.4.2 A*

In [ ]:
def astar_mapa(mapa,inicio,objetivo):
    tie=count(); frontera=[(manhattan(inicio,objetivo),0,next(tie),inicio,[inicio])]
    mejor_g={inicio:0}; expandidos=0
    while frontera:
        f,g,_,nodo,camino=heappop(frontera)
        if g != mejor_g.get(nodo): continue
        expandidos += 1
        if nodo==objetivo:
            return {"camino":camino,"costo":g,"expandidos":expandidos}
        for vecino in sucesores_mapa(mapa,nodo):
            ng=g+1
            if ng < mejor_g.get(vecino, math.inf):
                mejor_g[vecino]=ng
                nf=ng+manhattan(vecino,objetivo)
                heappush(frontera,(nf,ng,next(tie),vecino,camino+[vecino]))
    return None

resultado_astar_mapa=astar_mapa(mapa,INICIO,OBJETIVO)
resultado_astar_mapa

### 1.4.3 Visualización

In [ ]:
def mostrar_mapa(mapa, camino=None, inicio=None, objetivo=None, titulo="Mapa"):
    fig,ax=plt.subplots(figsize=(7,7))
    ax.imshow(mapa,cmap='cool')
    if camino:
        xs=[c for f,c in camino]; ys=[f for f,c in camino]
        ax.plot(xs,ys,c='red',marker="o")
    if inicio: ax.text(inicio[1],inicio[0],"Inicio",ha="center",va="center",fontsize=14)
    if objetivo: ax.text(objetivo[1],objetivo[0],"Final",ha="center",va="center",fontsize=14)
    ax.set_xticks(range(len(mapa[0]))); ax.set_yticks(range(len(mapa)))
    ax.set_title(titulo); ax.grid(True); plt.show()

mostrar_mapa(mapa,resultado_ucs_mapa["camino"],INICIO,OBJETIVO,"Costo Uniforme")
mostrar_mapa(mapa,resultado_astar_mapa["camino"],INICIO,OBJETIVO,"A* con Manhattan")

## 1.5 Beam Search

Beam Search limita deliberadamente la cantidad de estados conservados en cada nivel.

$$k=\text{beam width}$$

Si $k=2$, solo se mantienen los dos estados más prometedores del nivel.

Ventaja: reduce memoria y exploración.  
Riesgo: puede descartar un estado que conduzca a la mejor solución. Por eso no garantiza optimalidad y puede fallar aunque exista solución.

In [ ]:
def beam_search_mapa(mapa,inicio,objetivo,beam_width=2):
    haz=[(inicio,[inicio])]
    visitados={inicio}
    expandidos=0

    while haz:
        candidatos=[]
        for nodo,camino in haz:
            expandidos += 1
            if nodo==objetivo:
                return {"camino":camino,"costo":len(camino)-1,"expandidos":expandidos,"beam_width":beam_width}
            for vecino in sucesores_mapa(mapa,nodo):
                if vecino not in visitados:
                    visitados.add(vecino)
                    candidatos.append((vecino,camino+[vecino]))

        candidatos.sort(key=lambda x: manhattan(x[0],objetivo))
        haz=candidatos[:beam_width]
    return None

resultado_beam=beam_search_mapa(mapa,INICIO,OBJETIVO,beam_width=3)
resultado_beam

### 1.5.1 Experimentar con diferentes anchos de haz

In [ ]:
resultados_beam={}
for k in [1,2,3,5,10]:
    resultados_beam[k]=beam_search_mapa(mapa,INICIO,OBJETIVO,beam_width=k)
resultados_beam

## 1.6 Reto final: el 8-puzzle

Compare tres estrategias sobre el mismo problema:

1. BFS
2. A* con fichas fuera de lugar
3. A* con distancia Manhattan

Mida:
- longitud de la solución;
- estados expandidos;
- tiempo de ejecución.

Heurísticas sugeridas:

$$h_1(n)=\text{número de fichas fuera de lugar}$$

$$h_2(n)=\sum_i (|x_i-x_i^*|+|y_i-y_i^*|)$$

In [ ]:
estado_inicial = (
    1, 2, 3,
    4, 5, 0,
    6, 8, 7,
)

estado_objetivo = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0,
)

estado_inicial, estado_objetivo

### 1.6.1 Generación de sucesores

El `0` representa la casilla vacía.

La siguiente función genera todos los estados alcanzables mediante **un movimiento válido**.

In [ ]:
def sucesores_8_puzzle(estado):
    estado = list(estado)
    indice_cero = estado.index(0)

    fila, columna = divmod(indice_cero, 3)

    movimientos = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1),
    ]

    sucesores = []

    for df, dc in movimientos:
        nf, nc = fila + df, columna + dc

        if 0 <= nf < 3 and 0 <= nc < 3:
            nuevo_indice = nf * 3 + nc
            nuevo = estado.copy()

            nuevo[indice_cero], nuevo[nuevo_indice] = (
                nuevo[nuevo_indice],
                nuevo[indice_cero],
            )

            sucesores.append(tuple(nuevo))

    return sucesores


sucesores_8_puzzle(estado_inicial)

### 1.6.2 BFS del 8-puzzle

Referencia para poder comparar A* con una búsqueda no informada.

In [ ]:
from collections import deque

def bfs_8_puzzle(estado_inicial, estado_objetivo):
    cola = deque([(estado_inicial, [estado_inicial])])
    visitados = {estado_inicial}

    while cola:
        estado, camino = cola.popleft()

        for sucesor in sucesores_8_puzzle(estado):
            if sucesor not in visitados:
                visitados.add(sucesor)

                if sucesor == estado_objetivo:
                    return {
                        "camino": camino + [sucesor],
                        "movimientos": len(camino),
                        "explorados": len(visitados),
                    }

                cola.append((sucesor, camino + [sucesor]))

    return None


inicio_solucionable = (1, 2, 3, 4, 5, 0, 6, 7, 8)
resultado_bfs = bfs_8_puzzle(inicio_solucionable, estado_objetivo)

print("Movimientos:", resultado_bfs["movimientos"])
print("Estados explorados:", resultado_bfs["explorados"])

### 1.6.3 A* del 8-puzzle

Complete la implementación de A* para el 8-puzzle reutilizando `sucesores_8_puzzle`. Puede apoyarse en `ucs_mapa` y `astar_mapa`. Recuerde que el taller pide compararlo con BFS.

In [ ]:
# TODO: implemente aquí A* para el 8-puzzle.

def astar_8_puzzle(estado_inicial, estado_objetivo, heuristica):
    pass

# 2. Taller posterior

### Parte 1 — Cuadrícula
Construya un mapa de al menos `12 × 12` y ejecute:
- Costo Uniforme;
- A*;
- Beam Search con `k = 1, 2, 4, 8`.

Compare costo de solución, estados expandidos y capacidad para encontrar solución.

### Parte 2 — Heurísticas
Implemente dos heurísticas distintas para A* y analice cuál orienta mejor la búsqueda.

### Parte 3 — 8-puzzle
Implemente A* con fichas fuera de lugar y distancia Manhattan y compare con BFS.

### Pregunta final
**¿Puede una búsqueda más rápida producir una solución peor? Explique usando sus experimentos.**

### Uso de IA generativa
Si utiliza IA generativa, indique herramienta, propósito y partes de la actividad en las que fue utilizada. Debe poder explicar y defender completamente el trabajo entregado.

# 3. Solución del taller

## 3.1 Parte 1 — Cuadrícula

El enunciado pide construir un mapa de al menos `12 × 12` y ejecutar sobre él Costo Uniforme, A* y Beam Search con `k = 1, 2, 4, 8`, comparando costo de solución, estados expandidos y capacidad de encontrar una solución.

Construimos un mapa `12 × 12` con pasillos, cuellos de botella y un callejón sin salida en la columna izquierda. El inicio está en `(0, 0)` y el objetivo en `(11, 11)`; cada movimiento cuesta `1`.

In [ ]:
mapa12 = [
    [0,0,0,0,0,0,0,0,0,0,0,0],
    [0,1,1,1,1,0,0,0,1,1,1,0],
    [0,1,0,0,0,0,0,0,0,0,1,0],
    [0,1,0,1,1,1,1,1,1,0,1,0],
    [0,0,0,1,0,0,0,0,0,0,0,0],
    [0,1,1,1,0,1,1,1,1,1,1,0],
    [0,0,0,0,0,1,0,0,0,0,0,0],
    [0,1,1,1,1,1,0,1,1,1,1,0],
    [0,0,0,0,0,0,0,1,0,0,0,0],
    [0,1,1,0,1,1,1,1,0,1,1,0],
    [0,0,0,0,0,0,0,0,0,1,0,0],
    [0,0,1,0,0,0,0,0,0,1,0,0],
]

INICIO_12 = (0, 0)
OBJETIVO_12 = (11, 11)

print(f"Mapa {len(mapa12)} x {len(mapa12[0])}")
print(f"Inicio {INICIO_12}, objetivo {OBJETIVO_12}")
for fila in mapa12:
    print("".join("#" if celda == 1 else "." for celda in fila))

In [ ]:
resultados_parte1 = {}

resultados_parte1["Costo Uniforme"] = ucs_mapa(mapa12, INICIO_12, OBJETIVO_12)
resultados_parte1["A* Manhattan"] = astar_mapa(mapa12, INICIO_12, OBJETIVO_12)

for k in [1, 2, 4, 8]:
    resultados_parte1[f"Beam k={k}"] = beam_search_mapa(
        mapa12, INICIO_12, OBJETIVO_12, beam_width=k
    )

print(f"{'Algoritmo':<18} {'Solución':<9} {'Costo':<6} {'Expandidos'}")
print("-" * 46)
for nombre, res in resultados_parte1.items():
    if res is None:
        print(f"{nombre:<18} {'No':<9} {'-':<6} {'-':<10}")
    else:
        print(f"{nombre:<18} {'Sí':<9} {res['costo']:<6} {res['expandidos']:<10}")

In [ ]:
caminos_con_solucion = {
    nombre: res
    for nombre, res in resultados_parte1.items()
    if res is not None
}

for nombre, res in caminos_con_solucion.items():
    mostrar_mapa(
        mapa12,
        camino=res["camino"],
        inicio=INICIO_12,
        objetivo=OBJETIVO_12,
        titulo=nombre,
    )


Hicimos un mapa de `12 × 12` con un inicio en `(0, 0)`, un objetivo en `(11, 11)` y obstáculos que forman pasillos y cuellos de botella; entre ellos hay un callejón sin salida que sirve para ver cómo se comporta cada algoritmo. Reutilicé las funciones de las secciones 1.4 y 1.5.

### Resultados

| Algoritmo | ¿Encontró solución? | Costo | Estados expandidos |
|---|---:|---:|---:|
| Costo Uniforme | Sí | 22 | 93 |
| A* (Manhattan) | Sí | 22 | 73 |
| Beam k=1 | No | — | se atasca |
| Beam k=2 | Sí | 26 | 47 |
| Beam k=4 | Sí | 26 | 76 |
| Beam k=8 | Sí | 22 | 93 |

### Qué observé

- **Costo Uniforme:** siempre encuentra el camino más corto (22), pero revisa mucho (93) porque no sabe hacia dónde va el objetivo.
- **A*:** con Manhattan logra el mismo costo corto (22) y explorando menos (73), porque la heurística lo guía.
- **Beam k=1:** se pierde en el callejón sin salida y ni siquiera encuentra solución.
- **Beam k=2 y k=4:** sí llegan, pero con un camino más largo (26): podan demasiado y eliminan el buen camino.
- **Beam k=8:** conserva suficiente información y vuelve a encontrar el óptimo (22).

### Conclusión

Con un `k` pequeño Beam es más "rápido" (explora menos) pero puede llegar peor o ni siquiera llegar. La heurística bien usada (A*) logra rapidez sin perder calidad.